In [4]:
import pandas as pd
df = pd.read_excel('U SHAPE DATA.xlsx')
df.head()

,in:WIDTH,in:LENGTH,in:HIGHT,in:ORIENTATION,in:WINDOWS RATIO,out:FORM FACTOR,out:S/V RATIO,out:COOLING LOAD (KWH/M2),out:HEATING LOAD (KWH/M2),out:TOTAL LOAD (KWH/M2)
0,30,30,8,0,0.1,1.844920,0.461230,43.777210,1.765841,45.543051
1,40,30,8,0,0.1,1.829837,0.457459,43.911277,1.706487,45.617764
2,50,30,8,0,0.1,1.818182,0.454545,43.934824,1.678898,45.613723
3,60,30,8,0,0.1,1.808905,0.452226,43.912270,1.668411,45.580681
4,30,40,8,0,0.1,1.818182,0.454545,43.694254,1.806435,45.500689


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd


df['Energy_Class'] = pd.qcut(df['out:TOTAL LOAD (KWH/M2)'], q=3, labels=['Low', 'Medium', 'High'])


columns_to_drop = ['out:COOLING LOAD (KWH/M2)', 'out:HEATING LOAD (KWH/M2)', 'out:TOTAL LOAD (KWH/M2)', 'Energy_Class']
X = df.drop(columns_to_drop, axis=1)
y = df['Energy_Class']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print(f"Random Forest Accuracy (U Shape): {accuracy_score(y_test, rf_preds) * 100:.2f}%")
print(f"Random Forest F1-Score (U Shape): {f1_score(y_test, rf_preds, average='weighted') * 100:.2f}%\n")


le = LabelEncoder()
y_train_xgb = le.fit_transform(y_train)
y_test_xgb = le.transform(y_test)

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train_xgb)
xgb_preds = xgb_model.predict(X_test)

print(f"XGBoost Accuracy (U Shape): {accuracy_score(y_test_xgb, xgb_preds) * 100:.2f}%")
print(f"XGBoost F1-Score (U Shape): {f1_score(y_test_xgb, xgb_preds, average='weighted') * 100:.2f}%")

Random Forest Accuracy (U Shape): 85.39%
Random Forest F1-Score (U Shape): 85.43%

XGBoost Accuracy (U Shape): 94.48%
XGBoost F1-Score (U Shape): 94.53%


In [6]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Conv1D, Flatten, Input
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_aug, y_train_aug = smote.fit_resample(X_train_scaled, y_train_xgb)

print("Training Deep Learning Models for U Shape, please wait...\n")


ann_model = Sequential([
    Input(shape=(X_train_aug.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])
ann_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
ann_model.fit(X_train_aug, y_train_aug, epochs=50, batch_size=32, validation_data=(X_test_scaled, y_test_xgb), verbose=0)
ann_preds = np.argmax(ann_model.predict(X_test_scaled, verbose=0), axis=1)
print(f"ANN Accuracy (U Shape): {accuracy_score(y_test_xgb, ann_preds) * 100:.2f}%")
print(f"ANN F1-Score (U Shape): {f1_score(y_test_xgb, ann_preds, average='weighted') * 100:.2f}%\n")


X_train_rnn = X_train_aug.reshape((X_train_aug.shape[0], 1, X_train_aug.shape[1]))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

rnn_model = Sequential([
    Input(shape=(1, X_train_aug.shape[1])),
    SimpleRNN(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])
rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
rnn_model.fit(X_train_rnn, y_train_aug, epochs=50, batch_size=32, validation_data=(X_test_rnn, y_test_xgb), verbose=0)
rnn_preds = np.argmax(rnn_model.predict(X_test_rnn, verbose=0), axis=1)
print(f"RNN Accuracy (U Shape): {accuracy_score(y_test_xgb, rnn_preds) * 100:.2f}%")
print(f"RNN F1-Score (U Shape): {f1_score(y_test_xgb, rnn_preds, average='weighted') * 100:.2f}%\n")


cnn_model = Sequential([
    Input(shape=(1, X_train_aug.shape[1])),
    Conv1D(filters=32, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_rnn, y_train_aug, epochs=50, batch_size=32, validation_data=(X_test_rnn, y_test_xgb), verbose=0)
cnn_preds = np.argmax(cnn_model.predict(X_test_rnn, verbose=0), axis=1)
print(f"CNN Accuracy (U Shape): {accuracy_score(y_test_xgb, cnn_preds) * 100:.2f}%")
print(f"CNN F1-Score (U Shape): {f1_score(y_test_xgb, cnn_preds, average='weighted') * 100:.2f}%")

Training Deep Learning Models for U Shape, please wait...

ANN Accuracy (U Shape): 88.96%
ANN F1-Score (U Shape): 88.94%

RNN Accuracy (U Shape): 88.64%
RNN F1-Score (U Shape): 88.72%

CNN Accuracy (U Shape): 88.96%
CNN F1-Score (U Shape): 89.05%
